In [1]:
print("""
@File         : timezone_handling.ipynb
@Author(s)    : Stephen CUI
@LastEditor(s): Stephen CUI
@CreatedTime  : 2025-01-05 21:38:31
@Email        : cuixuanstephen@gmail.com
@Description  : 时区处理
""")


@File         : timezone_handling.ipynb
@Author(s)    : Stephen CUI
@LastEditor(s): Stephen CUI
@CreatedTime  : 2025-01-05 21:38:31
@Email        : cuixuanstephen@gmail.com
@Description  : 时区处理



In [2]:
import pandas as pd

To avoid issues when working with temporal data, it is critical to understand when you are working with timezone-aware datetimes (i.e., those tied to a timezone like UTC or America/New_York), and timezone-naive objects, which have no timezone information attached to them.

In [3]:
ser = pd.Series([
    "2024-01-01 00:00:00",
    "2024-01-02 00:00:01",
    "2024-01-03 00:00:02"
], dtype="datetime64[ns]")
ser

0   2024-01-01 00:00:00
1   2024-01-02 00:00:01
2   2024-01-03 00:00:02
dtype: datetime64[ns]

这些日期时间无法告诉我们这些事件发生的地点；纽约市的午夜与迪拜的午夜时间不同，因此很难确定这些事件发生的确切时间点。如果没有这些额外的元数据，这些日期时间就是时区无关的。

为了通过编程确认日期时间是否与时区无关，可以使用 `pd.Series.dt.tz`

In [4]:
ser.dt.tz is None

True

使用 `pd.Series.dt.tz_localize` 方法，我们可以为这些日期时间分配一个互联网号码分配机构(IANA)时区标识符，使它们具有时区感知能力。

In [12]:
ny_ser = ser.dt.tz_localize('America/New_York') # 为时间添加时区
ny_ser
# 这表明 ser 的时间是纽约的凌晨

0   2024-01-01 00:00:00-05:00
1   2024-01-02 00:00:01-05:00
2   2024-01-03 00:00:02-05:00
dtype: datetime64[ns, America/New_York]

In [13]:
ny_ser.dt.tz

<DstTzInfo 'America/New_York' LMT-1 day, 19:04:00 STD>

现在我们的 `pd.Series` 具有时区感知功能，其中包含的日期时间可以映射到世界任何地方的时间点。通过使用 `pd.Series.dt.tz_convert`，可以轻松地将这些事件转换为另一个时区：

In [14]:
la_ser = ny_ser.dt.tz_convert('America/Los_Angeles')
la_ser

0   2023-12-31 21:00:00-08:00
1   2024-01-01 21:00:01-08:00
2   2024-01-02 21:00:02-08:00
dtype: datetime64[ns, America/Los_Angeles]

实践中，最好将日期时间与时区关联起来，这样可以降低在不同日期或不同时间点被误解的风险。但是，并非所有与你交互的系统和数据库都能够保留此信息，因此你不得不放弃它以实现互操作性。如果有这样的需要，你可以通过将 `None` 作为参数传递给
 `pd.Series.dt.tz_localize` 来实现：

In [17]:
la_ser.dt.tz_localize(None)

0   2023-12-31 21:00:00
1   2024-01-01 21:00:01
2   2024-01-02 21:00:02
dtype: datetime64[ns]

如果被迫从日期时间中删除时区，我强烈建议将时区作为字符串存储在 pd.DataFrame 和数据库的另一列中：

In [20]:
df = la_ser.to_frame().assign(
    datetime=la_ser.dt.tz_localize(None), 
    timezone=str(la_ser.dt.tz)
).drop(columns=[0])

df

,datetime,timezone
0,2023-12-31 21:00:00,America/Los_Angeles
1,2024-01-01 21:00:01,America/Los_Angeles
2,2024-01-02 21:00:02,America/Los_Angeles


可以通过将时区列中的值应用于日期时间列中的数据来重新创建原始 pd.Series。

In [23]:
tz = df['timezone'].drop_duplicates().squeeze()
df['datetime'].dt.tz_localize(tz)

0   2023-12-31 21:00:00-08:00
1   2024-01-01 21:00:01-08:00
2   2024-01-02 21:00:02-08:00
Name: datetime, dtype: datetime64[ns, America/Los_Angeles]